In [1]:
#breathing frequency neurokit

# ── BREATHING FREQUENCY REPROCESSING — NEUROKIT2 ─────────────────────────────
import pandas as pd
import numpy as np
import neurokit2 as nk
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

bids_root = Path("C:/Users/alisa/OneDrive/Desktop/HRV_Biotrace/bids_data")
raw_path  = Path("C:/Users/alisa/OneDrive/Desktop/HRV_Biotrace/raw_biotrace_txt")
out_path  = bids_root / "derivatives" / "bf-analysis"
out_path.mkdir(parents=True, exist_ok=True)

SFREQ       = 32
SKIP_LINES  = 12
TASK_NAME   = "BBSIG"
PHASE_ORDER = ["pre", "early", "late", "post"]

participant_ids = [
    "001","002","003","004","005","006","007","008","009","010",
    "011","012","015","018","019","020","022","023","024","025"
]
session_ids = ["01","02"]

exclude_full = ["sub-021"]

# ── HELPER: load phase boundaries ─────────────────────────────────────────────
def load_phases(subj_id, ses_id):
    json_path = (bids_root / "derivatives" / "phases" /
                 ("sub-" + subj_id) / ("ses-" + ses_id) /
                 ("sub-" + subj_id + "_ses-" + ses_id +
                  "_task-" + TASK_NAME + "_physio_phases.json"))
    if not json_path.exists():
        return None
    with open(json_path) as f:
        return json.load(f)["phases"]

# ── HELPER: determine condition ────────────────────────────────────────────────
def get_condition(subj_id, ses_id):
    if (raw_path / ("sub" + subj_id + "_" + ses_id + "_c_bf.txt")).exists():
        return "control", "c"
    return "experimental", "e"

# ── HELPER: empty phase row ────────────────────────────────────────────────────
def empty_row(subj_key, ses_key, cond_code, phase_name):
    return {
        "subj_id":           subj_key,
        "ses_id":            ses_key,
        "condition":         cond_code,
        "phase":             phase_name,
        "BF_bpm":            np.nan,
        "BF_mean_amplitude": np.nan,
        "BF_n_cycles":       np.nan,
        "bf_dur_min":        np.nan,
        "bf_excluded":       True
    }

# ── MAIN BATCH LOOP ───────────────────────────────────────────────────────────
print("=" * 60)
print("BATCH: BREATHING FREQUENCY — NEUROKIT2 PIPELINE")
print("=" * 60)

all_results = []

for subj_id in participant_ids:
    for ses_id in session_ids:

        subj_key = "sub-" + subj_id
        ses_key  = "ses-" + ses_id
        id_key   = (subj_key, ses_key)

        # Skip fully excluded participants
        if subj_key in exclude_full:
            print("Skipping " + subj_key + "/" + ses_key + " (full exclusion)")
            continue

        cond_code, cond_letter = get_condition(subj_id, ses_id)
        print("\n[" + subj_key + "/" + ses_key + "] " + cond_code)

        # Load phase boundaries
        phases = load_phases(subj_id, ses_id)
        if phases is None:
            print("  WARNING: No phases JSON")
            continue

        # Check for missing BF file
        bf_file = raw_path / ("sub" + subj_id + "_" + ses_id + "_" + cond_letter + "_bf.txt")

        if not bf_file.exists() or id_key in exclude_bf:
            print("  WARNING: No BF file")
            for phase_name in PHASE_ORDER:
                all_results.append(empty_row(subj_key, ses_key, cond_code, phase_name))
            continue

        # Read raw BF signal
        try:
            raw = pd.read_csv(
                bf_file, sep="\t", skiprows=SKIP_LINES,
                header=None, names=["BF", "events", "empty"],
                encoding="utf-8"
            )
            bf_signal = pd.to_numeric(raw["BF"], errors="coerce").dropna().values
        except Exception as e:
            print("  WARNING: Read error: " + str(e))
            for phase_name in PHASE_ORDER:
                all_results.append(empty_row(subj_key, ses_key, cond_code, phase_name))
            continue

        # Reconstruct time axis from sample index
        time_s = np.arange(len(bf_signal)) / SFREQ

        # ── NEUROKIT2 RSP PROCESSING ──────────────────────────────────────────
        try:
            rsp_signals, rsp_info = nk.rsp_process(
                bf_signal,
                sampling_rate=SFREQ,
                method="khodadad2018"
            )
        except Exception as e:
            print("  WARNING: NeuroKit processing error: " + str(e))
            for phase_name in PHASE_ORDER:
                all_results.append(empty_row(subj_key, ses_key, cond_code, phase_name))
            continue

        # Peak and trough indices
        peaks_idx   = rsp_info["RSP_Peaks"]
        troughs_idx = rsp_info["RSP_Troughs"]

        overall_rate = len(peaks_idx) / (time_s[-1] / 60)
        print("  BF: " + str(len(peaks_idx)) + " peaks | " +
              str(round(overall_rate, 1)) + " br/min overall")

        # ── PER PHASE ─────────────────────────────────────────────────────────
        for phase_name in PHASE_ORDER:
            p       = phases[phase_name]
            start_s = p["start"]
            end_s   = p["end"]
            dur_min = (end_s - start_s) / 60

            # Peaks within phase window
            peak_times  = time_s[peaks_idx]
            phase_peaks = peaks_idx[
                (peak_times >= start_s) & (peak_times <= end_s)
            ]

            # Mean instantaneous rate from NeuroKit interpolated rate signal
            phase_mask        = (time_s >= start_s) & (time_s <= end_s)
            phase_rate_signal = rsp_signals.loc[phase_mask, "RSP_Rate"].values
            bf_bpm            = float(np.nanmean(phase_rate_signal))

            # Mean amplitude (tidal volume proxy)
            phase_amp = rsp_signals.loc[phase_mask, "RSP_Amplitude"].values
            mean_amp  = float(np.nanmean(phase_amp))

            all_results.append({
                "subj_id":           subj_key,
                "ses_id":            ses_key,
                "condition":         cond_code,
                "phase":             phase_name,
                "BF_bpm":            round(bf_bpm, 4),
                "BF_mean_amplitude": round(mean_amp, 4),
                "BF_n_cycles":       len(phase_peaks),
                "bf_dur_min":        round(dur_min, 4),
                "bf_excluded":       False
            })

            print("  " + phase_name.upper() +
                  ": " + str(round(bf_bpm, 1)) + " br/min" +
                  " | amp=" + str(round(mean_amp, 1)) +
                  " | n=" + str(len(phase_peaks)))

# ── SAVE OUTPUT ───────────────────────────────────────────────────────────────
df = pd.DataFrame(all_results)
out_file = out_path / ("task-" + TASK_NAME + "_per_phase_bf.tsv")
df.to_csv(out_file, sep="\t", index=False)

print("\n" + "=" * 60)
print("Saved: " + str(out_file.name))
print(str(len(df)) + " rows | " + str(df["BF_bpm"].isna().sum()) + " missing BF")
print("=" * 60)

# ── COMPARE WITH ORIGINAL CUSTOM METHOD ───────────────────────────────────────
old_file = bids_root / "derivatives/eda-analysis/task-BBSIG_eda-temp-bf_per_phase_final.tsv"

if old_file.exists():
    df_old = pd.read_csv(old_file, sep="\t")
    df_new = df[~df["bf_excluded"]].copy()

    comparison = df_new.merge(
        df_old[["subj_id", "ses_id", "condition", "phase", "BF_bpm"]].rename(
            columns={"BF_bpm": "BF_bpm_old"}
        ),
        on=["subj_id", "ses_id", "condition", "phase"],
        how="inner"
    )

    comparison["diff"] = comparison["BF_bpm"] - comparison["BF_bpm_old"]
    r = comparison[["BF_bpm", "BF_bpm_old"]].corr().iloc[0, 1]

    print("\nMETHOD COMPARISON (NeuroKit vs custom R):")
    print("  Mean BF NeuroKit : " + str(round(comparison["BF_bpm"].mean(), 2)) + " br/min")
    print("  Mean BF custom   : " + str(round(comparison["BF_bpm_old"].mean(), 2)) + " br/min")
    print("  Mean difference  : " + str(round(comparison["diff"].mean(), 3)) + " br/min")
    print("  Max abs diff     : " + str(round(comparison["diff"].abs().max(), 3)) + " br/min")
    print("  Correlation      : r=" + str(round(r, 4)))

    # Flag participants where methods disagree by more than 2 br/min
    large_diff = comparison[comparison["diff"].abs() > 2][
        ["subj_id", "ses_id", "phase", "BF_bpm", "BF_bpm_old", "diff"]
    ].sort_values("diff", key=abs, ascending=False)

    if len(large_diff) > 0:
        print("\n  Rows with abs difference > 2 br/min:")
        print(large_diff.to_string(index=False))
    else:
        print("\n  No rows with difference > 2 br/min - methods agree well")
else:
    print("\nOld BF file not found — skipping comparison")

BATCH: BREATHING FREQUENCY — NEUROKIT2 PIPELINE

[sub-001/ses-01] control


NameError: name 'exclude_bf' is not defined

In [3]:
#missing file per phase analysis
#Convert sub-018 ses-01 BF raw txt → BIDS + run BF per-phase analysis

import pandas as pd
import numpy as np
import neurokit2 as nk
import json, gzip, shutil
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ─── PATHS ────────────────────────────────────────────────────────────────────
RAW_FILE  = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\raw_biotrace_txt\sub_018_01_bf.txt")
BIDS_ROOT = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
BIDS_OUT  = BIDS_ROOT / "sub-018" / "ses-01" / "beh"
BF_TSV    = BIDS_ROOT / "derivatives" / "bf-analysis" / "task-BBSIG_per_phase_bf.tsv"
PHASES_JSON = (BIDS_ROOT / "derivatives" / "phases" /
               "sub-018" / "ses-01" /
               "sub-018_ses-01_task-BBSIG_physio_phases.json")

BIDS_OUT.mkdir(parents=True, exist_ok=True)

SFREQ      = 32          # Hz — confirmed from file header
CONDITION  = "experimental"
SUBJ_ID    = "sub-018"
SES_ID     = "ses-01"
TASK_NAME  = "BBSIG"
PHASE_ORDER = ["pre", "early", "late", "post"]

# ─── STEP 1: PARSE RAW SIGNAL ─────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Parsing raw BF signal")
print("=" * 60)

signal_vals = []
with open(RAW_FILE, encoding="latin-1") as f:
    for line in f:
        val = line.split("\t")[0].strip()
        # Skip header lines — only keep numeric values
        try:
            signal_vals.append(float(val))
        except ValueError:
            continue

signal = np.array(signal_vals)
time_s = np.arange(len(signal)) / SFREQ

print(f"  Samples   : {len(signal)}")
print(f"  Duration  : {time_s[-1]:.1f}s ({time_s[-1]/60:.1f} min)")
print(f"  Range     : {signal.min():.1f} – {signal.max():.1f}")

# ─── STEP 2: SAVE BIDS FILES ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Saving BIDS resp TSV.GZ + JSON")
print("=" * 60)

base_name = f"{SUBJ_ID}_{SES_ID}_task-{TASK_NAME}_resp"

# TSV (no header, single column = respiration)
tsv_path = BIDS_OUT / f"{base_name}.tsv"
pd.DataFrame(signal, columns=["respiration"]).to_csv(
    tsv_path, sep="\t", index=False, header=False)

# Compress to TSV.GZ
gz_path = BIDS_OUT / f"{base_name}.tsv.gz"
with open(tsv_path, "rb") as f_in, gzip.open(gz_path, "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)
tsv_path.unlink()  # remove uncompressed version
print(f"  Saved → {gz_path.name}")

# JSON sidecar
json_meta = {
    "SamplingFrequency": SFREQ,
    "StartTime": 0.0,
    "Columns": ["respiration"],
    "Units": "ARU",
    "Manufacturer": "Mind Media",
    "ManufacturersModelName": "BioTrace+",
    "RecordingType": "continuous",
    "PhysiologicalRecordingType": "Respiration",
    "Condition": CONDITION,
    "TaskName": "Audiovisual Relaxation",
    "TaskDescription": (
        "Participants were exposed to either stroboscopic light and "
        "binaural beats (experimental) or constant light and music (control)"
    )
}
json_path = BIDS_OUT / f"{base_name}.json"
with open(json_path, "w") as f:
    json.dump(json_meta, f, indent=4)
print(f"  Saved → {json_path.name}")

# ─── STEP 3: NEUROKIT2 RSP PROCESSING ────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — NeuroKit2 RSP processing (khodadad2018)")
print("=" * 60)

rsp_signals, rsp_info = nk.rsp_process(
    signal, sampling_rate=SFREQ, method="khodadad2018")

peaks_idx = rsp_info["RSP_Peaks"]
overall_rate = len(peaks_idx) / (time_s[-1] / 60)
print(f"  Peaks detected : {len(peaks_idx)}")
print(f"  Overall BF     : {overall_rate:.1f} br/min")

# ─── STEP 4: LOAD PHASES + EXTRACT PER-PHASE BF ──────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Per-phase BF extraction")
print("=" * 60)

if not PHASES_JSON.exists():
    print(f"  ⚠  Phases JSON not found: {PHASES_JSON}")
    print("     Cannot extract per-phase values.")
    raise FileNotFoundError(PHASES_JSON)

with open(PHASES_JSON) as f:
    phases = json.load(f)["phases"]

new_rows = []
for phase_name in PHASE_ORDER:
    p       = phases[phase_name]
    start_s = p["start"]
    end_s   = p["end"]
    dur_min = (end_s - start_s) / 60

    # Mean instantaneous rate from NeuroKit interpolated signal
    phase_mask        = (time_s >= start_s) & (time_s <= end_s)
    phase_rate_signal = rsp_signals.loc[phase_mask, "RSP_Rate"].values
    bf_bpm            = float(np.nanmean(phase_rate_signal))

    # Mean amplitude
    phase_amp = rsp_signals.loc[phase_mask, "RSP_Amplitude"].values
    mean_amp  = float(np.nanmean(phase_amp))

    # Peaks in phase
    peak_times  = time_s[peaks_idx]
    phase_peaks = peaks_idx[
        (peak_times >= start_s) & (peak_times <= end_s)]

    new_rows.append({
        "subj_id":           SUBJ_ID,
        "ses_id":            SES_ID,
        "condition":         CONDITION,
        "phase":             phase_name,
        "BF_bpm":            round(bf_bpm, 4),
        "BF_mean_amplitude": round(mean_amp, 4),
        "BF_n_cycles":       len(phase_peaks),
        "bf_dur_min":        round(dur_min, 4),
        "bf_excluded":       False
    })

    print(f"  {phase_name.upper():6s}: {bf_bpm:.1f} br/min | "
          f"amp={mean_amp:.1f} | n_peaks={len(phase_peaks)} | "
          f"dur={dur_min:.2f} min")

df_new = pd.DataFrame(new_rows)

# ─── STEP 5: PATCH INTO MAIN BF TSV ──────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Patching into main BF TSV")
print("=" * 60)

df_bf = pd.read_csv(BF_TSV, sep="\t")
print(f"  Rows before patch: {len(df_bf)}")

# Show what's currently there for sub-018 ses-01
existing = df_bf[
    (df_bf["subj_id"] == SUBJ_ID) &
    (df_bf["ses_id"]  == SES_ID)]
print(f"  Existing sub-018 ses-01 rows: {len(existing)}")
if not existing.empty:
    print(existing[["subj_id","ses_id","condition",
                     "phase","BF_bpm"]].to_string(index=False))

# Remove existing NaN rows for sub-018 ses-01 and replace with new values
df_bf = df_bf[~(
    (df_bf["subj_id"] == SUBJ_ID) &
    (df_bf["ses_id"]  == SES_ID)
)]

# Add new columns if they don't exist yet
for col in ["BF_mean_amplitude", "BF_n_cycles", "bf_dur_min", "bf_excluded"]:
    if col not in df_bf.columns:
        df_bf[col] = np.nan

df_bf = pd.concat([df_bf, df_new], ignore_index=True)

# Sort to keep logical order
df_bf = df_bf.sort_values(["subj_id","ses_id","phase"]).reset_index(drop=True)

df_bf.to_csv(BF_TSV, sep="\t", index=False)
print(f"\n  Rows after patch : {len(df_bf)}")
print(f"\n  New sub-018 ses-01 values:")
print(df_bf[
    (df_bf["subj_id"] == SUBJ_ID) &
    (df_bf["ses_id"]  == SES_ID)
][["subj_id","ses_id","condition","phase","BF_bpm"]].to_string(index=False))

print(f"\n  ✅ Patched BF TSV saved → {BF_TSV.name}")
# print("=" * 60)

STEP 1 — Parsing raw BF signal
  Samples   : 58208
  Duration  : 1819.0s (30.3 min)
  Range     : 951.2 – 1177.0

STEP 2 — Saving BIDS resp TSV.GZ + JSON
  Saved → sub-018_ses-01_task-BBSIG_resp.tsv.gz
  Saved → sub-018_ses-01_task-BBSIG_resp.json

STEP 3 — NeuroKit2 RSP processing (khodadad2018)
  Peaks detected : 580
  Overall BF     : 19.1 br/min

STEP 4 — Per-phase BF extraction
  PRE   : 16.0 br/min | amp=27.9 | n_peaks=27 | dur=1.77 min
  EARLY : 19.4 br/min | amp=4.0 | n_peaks=49 | dur=2.50 min
  LATE  : 19.7 br/min | amp=4.6 | n_peaks=59 | dur=3.00 min
  POST  : 19.7 br/min | amp=10.3 | n_peaks=58 | dur=3.00 min

STEP 5 — Patching into main BF TSV
  Rows before patch: 160
  Existing sub-018 ses-01 rows: 4
subj_id ses_id    condition phase  BF_bpm
sub-018 ses-01 experimental early 19.4046
sub-018 ses-01 experimental  late 19.7333
sub-018 ses-01 experimental  post 19.6542
sub-018 ses-01 experimental   pre 16.0326

  Rows after patch : 160

  New sub-018 ses-01 values:
subj_id ses

In [5]:
#calculate heart frequency
import pandas as pd
import numpy as np
import neurokit2 as nk
import json
from pathlib import Path

bids_root = Path("C:/Users/alisa/OneDrive/Desktop/HRV_Biotrace/bids_data")
out_path  = bids_root / "derivatives" / "hrv-analysis"
out_path.mkdir(parents=True, exist_ok=True)

participant_ids = [
    "001","002","003","004","005","006","007","008","009","010",
    "011","012","015","018","019","020","022","023","024","025"
]
session_ids = ["01","02"]
TASK_NAME   = "BBSIG"
PHASE_ORDER = ["pre","early","late","post"]

all_rows = []

for subj_id in participant_ids:
    for ses_id in session_ids:
        subj_key = "sub-" + subj_id
        ses_key  = "ses-" + ses_id

        # ── LOAD BBSIG ECG PREPROC JSON ───────────────────────────────────────
        json_path = (
            bids_root / "derivatives" / "ecg-preproc" /
            subj_key / ses_key /
            f"{subj_key}_{ses_key}_task-{TASK_NAME}_physio_ecg-preproc.json"
        )
        if not json_path.exists():
            print(f"Missing: {subj_key}/{ses_key}")
            continue

        with open(json_path) as f:
            ecg_data = json.load(f)

        FS_ECG = ecg_data["info"]["sampling_rate"]

        # Use corrected peaks if available, otherwise original
        rpeaks_dict = ecg_data["rpeaks"]
        rpeaks_samples = np.array(
            rpeaks_dict.get("ECG_R_Peaks_corrected",
            rpeaks_dict.get("ECG_R_Peaks_original", []))
        )

        rr_dict = ecg_data["rr_s"]
        rr_s = np.array(
            rr_dict.get("RR_s_corrected",
            rr_dict.get("RR_s_original", []))
        )

        # Total recording length in samples
        # estimated from last R-peak + one more RR interval
        total_samples = int(rpeaks_samples[-1] + rr_s[-1] * FS_ECG)

        # ── COMPUTE CONTINUOUS HR USING nk.ecg_rate() ────────────────────────
        # ecg_rate() = signal_rate() alias in NeuroKit2
        # Computes 60 / inter-peak period, interpolated with monotone cubic
        # spline across desired_length samples (Makowski et al., 2021)
        ecg_rate = nk.ecg_rate(
            peaks                = rpeaks_samples,
            sampling_rate        = FS_ECG,
            desired_length       = total_samples,
            interpolation_method = "monotone_cubic"
        )

        # ── LOAD PHASE BOUNDARIES ─────────────────────────────────────────────
        phases_path = (
            bids_root / "derivatives" / "phases" /
            subj_key / ses_key /
            f"{subj_key}_{ses_key}_task-{TASK_NAME}_physio_phases.json"
        )
        if not phases_path.exists():
            print(f"Missing phases: {subj_key}/{ses_key}")
            continue

        with open(phases_path) as f:
            phases_json = json.load(f)

        phases    = phases_json.get("phases", phases_json)
        condition = phases_json.get("condition", "unknown")

        # ── AVERAGE ECG_Rate WITHIN EACH PHASE ───────────────────────────────
        for phase_name in PHASE_ORDER:
            if phase_name not in phases:
                continue

            p         = phases[phase_name]
            start_idx = int(float(p["start"]) * FS_ECG)
            end_idx   = min(int(float(p["end"]) * FS_ECG), total_samples - 1)

            phase_rate = ecg_rate[start_idx:end_idx]
            phase_rate = phase_rate[~np.isnan(phase_rate)]

            if len(phase_rate) < 10:
                print(f"  Too few samples: {subj_key}/{ses_key}/{phase_name} "
                      f"(n={len(phase_rate)})")
                hr_mean = hr_sd = hr_min = hr_max = np.nan
            else:
                hr_mean = np.mean(phase_rate)
                hr_sd   = np.std(phase_rate, ddof=1)
                hr_min  = np.min(phase_rate)
                hr_max  = np.max(phase_rate)

            all_rows.append({
                "subj_id":   subj_key,
                "ses_id":    ses_key,
                "condition": condition,
                "phase":     phase_name,
                "HR_mean":   round(hr_mean, 3) if not np.isnan(hr_mean) else np.nan,
                "HR_sd":     round(hr_sd,   3) if not np.isnan(hr_sd)   else np.nan,
                "HR_min":    round(hr_min,  3) if not np.isnan(hr_min)  else np.nan,
                "HR_max":    round(hr_max,  3) if not np.isnan(hr_max)  else np.nan,
                "n_samples": len(phase_rate)
            })

# ── OUTPUT ────────────────────────────────────────────────────────────────────
df_hr = pd.DataFrame(all_rows)

out_file = out_path / "task-BBSIG_per_phase_hr_nk.tsv"
df_hr.to_csv(out_file, sep="\t", index=False)

print("\n" + "=" * 60)
print("HEART RATE SUMMARY — nk.ecg_rate()")
print("=" * 60)
print(f"\nTotal records: {len(df_hr)}")
print(f"Missing HR:    {df_hr['HR_mean'].isna().sum()}")

print("\nMean HR per phase (bpm):")
print(df_hr.groupby("phase")["HR_mean"]
      .agg(["mean","std","min","max"])
      .round(2)
      .loc[PHASE_ORDER])

print("\nMean HR per condition:")
print(df_hr.groupby("condition")["HR_mean"]
      .agg(["mean","std"])
      .round(2))

print(f"\nSaved: {out_file.name}")


HEART RATE SUMMARY — nk.ecg_rate()

Total records: 160
Missing HR:    0

Mean HR per phase (bpm):
        mean    std    min     max
phase                             
pre    75.48  12.57  56.27  106.56
early  71.39  11.74  49.82  100.00
late   69.87  12.10  46.49   95.00
post   77.46  10.41  59.40   98.34

Mean HR per condition:
               mean    std
condition                 
control       73.09  11.40
experimental  74.00  12.66

Saved: task-BBSIG_per_phase_hr_nk.tsv


In [2]:
#structure of bid view

import json
from pathlib import Path

bids_root = Path("C:/Users/alisa/OneDrive/Desktop/HRV_Biotrace/bids_data")
ecg_path  = bids_root / "derivatives" / "ecg-preproc"

json_path = (ecg_path / "sub-001" / "ses-01" /
             "sub-001_ses-01_task-BBSIG_physio_ecg-preproc.json")

with open(json_path) as f:
    ecg_data = json.load(f)

print("Top-level keys:", list(ecg_data.keys()))
print("\nType of rpeaks:", type(ecg_data["rpeaks"]))
print("Type of rr_s:",   type(ecg_data["rr_s"]))
print("\nrpeaks preview:", str(ecg_data["rpeaks"])[:200])
print("\nrr_s preview:",   str(ecg_data["rr_s"])[:200])
print("\ninfo:",           ecg_data.get("info"))

Top-level keys: ['rpeaks', 'qrs', 'rr_s', 'info']

Type of rpeaks: <class 'dict'>
Type of rr_s: <class 'dict'>

rpeaks preview: {'ECG_R_Peaks_original': [209, 409, 609, 807, 1005, 1203, 1400, 1599, 1797, 1997, 2193, 2387, 2559, 2731, 2957, 3150, 3341, 3532, 3725, 3922, 4119, 4315, 4515, 4714, 4911, 5109, 5304, 5498, 5692, 5886

rr_s preview: {'RR_s_original': [0.78125, 0.78125, 0.7734375, 0.7734375, 0.7734375, 0.76953125, 0.77734375, 0.7734375, 0.78125, 0.765625, 0.7578125, 0.671875, 0.671875, 0.8828125, 0.75390625, 0.74609375, 0.74609375

info: {'rpeak_source': 'original', 'sampling_rate': 256, 'n_peaks': 2279, 'delineation_method': 'dwt'}
